# Week 21: Professional testing, code quality, and continuous integration

Month 6 is about turning a working project into one that **someone else can trust, run and extend**. It's the difference between "student project" and "hire this person".

| Session | Topic | Time |
|---|---|---|
| Day 1 | pytest beyond basics: fixtures, `conftest.py`, parametrize, testing ML code | 2.5 h |
| Day 2 | Coverage: what isn't tested? | 2 h |
| Day 3 | Linting, formatting, type hints, docstrings (ruff) | 2.5 h |
| Day 4 | GitHub Actions: tests run on every push | 2.5 h |
| Review | Spaced recall | 0.5 h |

**Install:** `pip install -e ".[dev]"` (pytest-cov, ruff).

### Done when
- [ ] Shared fixtures live in `tests/conftest.py`; at least one parametrized test; at least one property-style test.
- [ ] Coverage of the core modules (metrics, ranking, fusion, stats, report, imaging metrics, calibration) is ≥ 80%.
- [ ] `ruff check medlab tests` is clean.
- [ ] CI is green on GitHub, with a badge in the README.

In [ ]:
import sys, importlib, subprocess
from pathlib import Path
for p in ["..", "../capstone"]:
    if str(Path(p).resolve()) not in sys.path:
        sys.path.insert(0, str(Path(p).resolve()))
CAP = Path("../capstone")

def run_tests(*paths):
    """Run pytest inside the capstone folder and return its exit code (0 = all passed)."""
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", *paths], cwd=CAP, capture_output=True, text=True)
    print(r.stdout[-2500:], r.stderr[-800:])
    return r.returncode

def fresh(module_name):
    """Import (or re-import after you edited it) a module from your capstone package."""
    importlib.invalidate_caches()          # notice files you created after the kernel started
    return importlib.reload(importlib.import_module(module_name))

---
# Day 1: Better tests (2.5 h)

**Fixtures** build test inputs once and share them: put them in `tests/conftest.py`, and pytest injects them into any test that names them as an argument.
```python
# tests/conftest.py
import pytest
from medlab.index import InvertedIndex
from medlab.text import preprocess

@pytest.fixture
def tiny_docs():
    return {"d1": "Vitamin D and bone health", "d2": "Statins lower cholesterol", "d3": "Bone density in older adults"}

@pytest.fixture
def tiny_index(tiny_docs):           # fixtures can use other fixtures
    return InvertedIndex.build(tiny_docs, preprocess)
```
**Parametrize** runs one test on many inputs:
```python
@pytest.mark.parametrize("k", [1, 3, 10])
def test_precision_bounded(k): ...
```
**Testing ML code**: you rarely know the exact "right" output, so test **properties** instead:
- *bounds*: every metric is in [0, 1];
- *invariances*: shuffling the document order doesn't change BM25 scores; adding an irrelevant document doesn't change a query's top result unless it matches;
- *known cases*: a perfect ranking gives nDCG = 1; identical systems give p = 1;
- *shapes and dtypes*: the model output is (N, 2) float32;
- *determinism*: same seed, same result.

### Exercise 21.1
Create `tests/conftest.py` with at least 2 fixtures, and refactor at least one existing test file to use them. Add `tests/test_properties.py` with **at least 4 property tests**, including at least one `@pytest.mark.parametrize` and at least one using random inputs (seeded).

In [ ]:
# Write conftest.py and test_properties.py, then run the checker.

In [ ]:
conf = (CAP / "tests" / "conftest.py").read_text()
props = (CAP / "tests" / "test_properties.py").read_text()
assert conf.count("@pytest.fixture") >= 2
assert "parametrize" in props and props.count("def test_") >= 4
assert run_tests("tests/test_properties.py") == 0
print("✅ 21.1 passed")

---
# Day 2: Coverage (2 h)

Coverage measures which lines your tests execute. **100% coverage doesn't mean correct**, but uncovered lines are *definitely* untested.
```bash
python -m pytest --cov=medlab --cov-report=term-missing
```
The `Missing` column lists the line numbers that never ran. Aim for **≥ 80% on the core logic**. Don't chase coverage of CLI glue or plotting.

### Exercise 21.2
Add tests until the core modules reach 80% combined. The checker measures exactly these modules.

In [ ]:
# Add tests, then run the checker.

In [ ]:
import subprocess, sys, re
core = ["medlab/metrics.py", "medlab/ranking.py", "medlab/fusion.py", "medlab/stats.py", "medlab/report.py",
        "medlab/imaging/metrics.py", "medlab/imaging/calibration.py"]
out = subprocess.run([sys.executable, "-m", "pytest", "-q", "--cov=medlab", "--cov-report=term", "-p", "no:cacheprovider"],
                     cwd=CAP, capture_output=True, text=True)
lines = [l for l in out.stdout.splitlines() if any(l.startswith(c) for c in core)]
print("\n".join(lines))
stm = sum(int(l.split()[1]) for l in lines); miss = sum(int(l.split()[2]) for l in lines)
cov = 1 - miss / stm
print(f"core coverage: {cov:.1%}")
assert out.returncode == 0 and cov >= 0.80
print("✅ 21.2 passed")

---
# Day 3: Linting, formatting, types (2.5 h)

- **Formatter** (`ruff format`): consistent style, so no arguments about it in code review.
- **Linter** (`ruff check`): catches unused imports, undefined names, bad comparisons, mutable default arguments (Week 1, bug 2!).
- **Type hints**: `def ndcg_at_k(ranked: list[str], rel: dict[str, int], k: int) -> float:`. They're documentation that tools can check.
- **Docstrings**: one line on what the function does, plus arguments and returns if it isn't obvious.

Add this to `pyproject.toml`:
```toml
[tool.ruff]
line-length = 110
[tool.ruff.lint]
select = ["E", "F", "I", "B", "UP"]      # pycodestyle, pyflakes, import sorting, bugbear, pyupgrade
ignore = ["E501", "E731", "E402"]       # line length is the formatter's job; allow small lambdas;
                                        # allow imports after matplotlib.use("Agg")
```
### Exercise 21.3
Run `ruff format .` and `ruff check --fix .`, then fix the rest by hand. Add type hints and docstrings to every public function in `metrics.py`, `ranking.py` and `stats.py`.

In [ ]:
# Configure ruff and clean the code, then run the checker.

In [ ]:
out = subprocess.run(["ruff", "check", "medlab", "tests"], cwd=CAP, capture_output=True, text=True)
print(out.stdout[-2000:])
assert out.returncode == 0, "ruff still reports problems"
import ast
for f in ["medlab/metrics.py", "medlab/ranking.py", "medlab/stats.py"]:
    tree = ast.parse((CAP / f).read_text())
    for fn in [n for n in ast.walk(tree) if isinstance(n, ast.FunctionDef) and not n.name.startswith("_")]:
        assert fn.returns is not None, f"{f}:{fn.name} has no return type hint"
        assert ast.get_docstring(fn), f"{f}:{fn.name} has no docstring"
print("✅ 21.3 passed")

---
# Day 4: Continuous integration with GitHub Actions (2.5 h)

**CI** runs your tests automatically on every push and pull request, on a clean machine. It catches "works on my laptop" bugs and missing dependencies. Create `.github/workflows/ci.yml`:
```yaml
name: CI
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    env:
      COURSE_FAKE_DATA: "1"          # tests use synthetic data: fast, no downloads
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: pip
      - name: Install
        run: |
          pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
          pip install -e ".[dev]"
      - name: Lint
        run: ruff check medlab tests
      - name: Test
        run: python -m pytest -q --cov=medlab
```
Push it, open the **Actions** tab on GitHub, and watch it run. When it's green, add the badge to your README:
`![CI](https://github.com/<you>/medsearch-lab/actions/workflows/ci.yml/badge.svg)`

**If CI fails but your laptop passes**, the cause is almost always one of: a missing dependency in `pyproject.toml`, a test that needs a downloaded file, a hard-coded absolute path, or a test that depends on another test's leftovers. All four are real bugs, so fix them.

### Exercise 21.4: Create the workflow file

In [ ]:
# Create .github/workflows/ci.yml, push it, then run the checker.

In [ ]:
import yaml
wf = yaml.safe_load((CAP / ".github" / "workflows" / "ci.yml").read_text())
job = next(iter(wf["jobs"].values()))
runs = " ".join(s.get("run", "") for s in job["steps"])
assert "pytest" in runs and "ruff" in runs
assert str(job.get("env", {}).get("COURSE_FAKE_DATA")) == "1"
print("✅ 21.4 passed. Now push, and check that the Actions tab is green.")

---
# Review (30 min)
1. What's the difference between a fixture and a helper function?
2. Name three properties you can test without knowing the exact output of a model.
3. Why doesn't 100% coverage mean correct code?
4. *(Week 1)* Which ruff rule family would have caught the mutable default argument bug?
5. *(Week 17)* Why does CI install with `pip install -e ".[dev]"` rather than from your `requirements.txt`?
6. *(Week 19)* Why choose the threshold on validation?

✍️ *Your answer:* (double-click to edit)

